In [3]:
"""
Comprehensive multi-problem benchmark: AGS v3.7 vs Optuna TPE.

5 estimator families, 6 datasets, 2 budgets, 5 seeds.
Per-problem tables + per-estimator summary + final aggregate + niche analysis.

Run:
    python comprehensive_benchmark.py
"""
import time, warnings
from statistics import mean
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer, load_wine, load_digits, load_iris
from sklearn.datasets import make_classification, make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import optuna
from ags import AdaptiveGreedySearch

optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_XGB = False

try:
    from scipy.stats import wilcoxon
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False


# ============================================================
# CONFIG
# ============================================================
CV          = 5
BUDGETS     = [30, 60]
N_SEEDS     = 5
N_JOBS      = -1
SCORING     = "f1_macro"
SEED_BASE   = 0


# ============================================================
# ROBUST AGS ATTRIBUTE ACCESS
# ============================================================
def ags_grab(obj, names, default=None):
    for n in names:
        if hasattr(obj, n):
            v = getattr(obj, n)
            if v is not None:
                return v
    return default


def ags_extract(ags):
    score = ags_grab(ags, ["best_score_", "best_score",
                           "best_value_", "best_value", "best_fitness"])
    saved = ags_grab(ags, ["folds_saved", "saved_folds",
                           "folds_pruned", "pruned_folds"], 0)
    poss  = ags_grab(ags, ["total_folds_possible", "folds_possible",
                           "total_folds", "n_folds_total"], 0)
    run   = ags_grab(ags, ["total_folds_run", "folds_run",
                           "n_folds_run", "folds_executed"], None)
    return score, int(saved or 0), int(poss or 0), run


# ============================================================
# DATASETS
# ============================================================
def _load(name):
    if name == "breast_cancer":
        d = load_breast_cancer(); return d.data, d.target
    if name == "wine":
        d = load_wine(); return d.data, d.target
    if name == "digits":
        d = load_digits(); return d.data, d.target
    if name == "iris":
        d = load_iris(); return d.data, d.target
    if name == "synthetic_high_dim":
        return make_classification(
            n_samples=600, n_features=25, n_informative=12,
            n_classes=3, n_clusters_per_class=2, random_state=0)
    if name == "moons":
        return make_moons(n_samples=600, noise=0.25, random_state=0)
    raise ValueError(name)


# ============================================================
# SEARCH SPACES
# ============================================================
def rf_space():
    return (
        RandomForestClassifier(random_state=0, n_jobs=1),
        {
            "n_estimators":      [50, 100, 200, 300],
            "max_depth":         [3, 5, 7, 10, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf":  [1, 2, 4],
            "max_features":      ["sqrt", "log2"],
        },
        "discrete/ordinal",
    )


def xgb_space():
    if HAS_XGB:
        # Build XGBClassifier without the removed `use_label_encoder` kwarg
        try:
            model = XGBClassifier(
                random_state=0, n_jobs=1, verbosity=0, eval_metric="logloss",
            )
        except TypeError:
            model = XGBClassifier(random_state=0, n_jobs=1, verbosity=0)
        grid = {
            "max_depth":        [3, 5, 7],
            "learning_rate":    [0.01, 0.05, 0.1, 0.3],
            "n_estimators":     [50, 100, 200],
            "subsample":        [0.6, 0.8, 1.0],
            "colsample_bytree": [0.6, 0.8, 1.0],
        }
    else:
        model = HistGradientBoostingClassifier(random_state=0)
        grid = {
            "max_depth":          [3, 5, 7, None],
            "learning_rate":      [0.01, 0.05, 0.1, 0.3],
            "max_iter":           [50, 100, 200],
            "min_samples_leaf":   [10, 20, 30],
            "l2_regularization":  [0.0, 0.1, 1.0],
        }
    return model, grid, "mixed continuous/discrete"


def svm_space():
    return (
        make_pipeline(StandardScaler(), SVC(random_state=0)),
        {
            "svc__C":      [0.01, 0.1, 1, 10, 100],
            "svc__gamma":  ["scale", 0.001, 0.01, 0.1],
            "svc__kernel": ["linear", "rbf", "poly"],
            "svc__degree": [2, 3, 4],
        },
        "conditional (flattened)",
    )


def mlp_space():
    return (
        make_pipeline(
            StandardScaler(),
            MLPClassifier(random_state=0, max_iter=500, early_stopping=True),
        ),
        {
            "mlpclassifier__hidden_layer_sizes": [(32,), (64,), (32, 32), (64, 32), (64, 64)],
            "mlpclassifier__alpha":              [0.0001, 0.001, 0.01, 0.1],
            "mlpclassifier__learning_rate_init": [0.001, 0.01],
        },
        "strongly interacting",
    )


def knn_space():
    return (
        KNeighborsClassifier(),
        {
            "n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
            "weights":     ["uniform", "distance"],
            "p":           [1, 2],
            "leaf_size":   [20, 30, 40],
            "algorithm":   ["auto", "ball_tree", "kd_tree"],
        },
        "simple",
    )


ESTIMATORS = {
    "RandomForest": rf_space,
    "XGBoost" if HAS_XGB else "HistGBM": xgb_space,
    "SVM": svm_space,
    "MLP": mlp_space,
    "KNN": knn_space,
}


# ============================================================
# PROBLEM SET
# ============================================================
PROBLEMS = [
    ("breast_cancer",     "RandomForest"),
    ("breast_cancer",     "SVM"),
    ("wine",              "RandomForest"),
    ("wine",              "SVM"),
    ("digits",            "RandomForest"),
    ("digits",            "MLP"),
    ("iris",              "KNN"),
    ("synthetic_high_dim","XGBoost" if HAS_XGB else "HistGBM"),
    ("moons",             "SVM"),
    ("digits",            "KNN"),
]


# ============================================================
# RUNNERS
# ============================================================
def run_optuna(model, grid, X, y, budget, seed):
    t0 = time.perf_counter()

    def objective(trial):
        p = {k: trial.suggest_categorical(k, list(v)) for k, v in grid.items()}
        m = clone(model)                    # <-- correct Pipeline-safe clone
        m.set_params(**p)
        return float(cross_val_score(
            m, X, y, cv=CV, scoring=SCORING,
            n_jobs=N_JOBS, error_score="raise",
        ).mean())

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=budget, show_progress_bar=False)
    return float(study.best_value), time.perf_counter() - t0


def run_ags(model, grid, X, y, budget, seed):
    t0 = time.perf_counter()
    ags = AdaptiveGreedySearch(
        estimator=model,
        param_grid=grid,
        cv=CV,
        scoring=SCORING,
        max_evaluations=budget,
        n_jobs=N_JOBS,
        random_state=seed,
    )
    ags.fit(X, y)
    elapsed = time.perf_counter() - t0
    score, saved, poss, run = ags_extract(ags)
    if score is None:
        raise AttributeError(
            "Could not extract score from AGS object. "
            f"Available public attrs: {[a for a in dir(ags) if not a.startswith('_')]}"
        )
    return float(score), elapsed, saved, poss, run


# ============================================================
# MAIN
# ============================================================
print("=" * 92)
print("COMPREHENSIVE MULTI-PROBLEM BENCHMARK — AGS v3.7 vs Optuna TPE")
print(f"Metric: {SCORING}  |  CV: {CV}  |  Seeds: {N_SEEDS}  |  Budgets: {BUDGETS}")
print(f"n_jobs: {N_JOBS} (both)  |  XGBoost available: {HAS_XGB}")
print("=" * 92)

# Probe: confirm AGS attributes are extractable before the real run
_probe_model, _probe_grid, _ = rf_space()
try:
    _Xp, _yp = _load("iris")
    _probe = AdaptiveGreedySearch(
        estimator=_probe_model, param_grid=_probe_grid,
        cv=CV, scoring=SCORING, max_evaluations=2,
        n_jobs=1, random_state=0,
    )
    _probe.fit(_Xp, _yp)
    _public = [a for a in dir(_probe) if not a.startswith("_")]
    print(f"AGS public attributes (probe): {_public}")
    _score, _saved, _poss, _run = ags_extract(_probe)
    print(f"Extracted -> score={_score}  folds_saved={_saved}  "
          f"folds_possible={_poss}  folds_run={_run}")
except Exception as _e:
    print(f"AGS probe failed: {type(_e).__name__}: {_e}")
print("=" * 92)

all_results = []

for ds_name, est_name in PROBLEMS:
    X, y = _load(ds_name)
    model, grid, space_type = ESTIMATORS[est_name]()
    grid_size = int(np.prod([len(v) for v in grid.values()]))

    print(f"\n{'='*92}")
    print(f"PROBLEM: {ds_name}  ×  {est_name}  [{space_type}]")
    print(f"  Data: {X.shape[0]} samples × {X.shape[1]} features, "
          f"{len(np.unique(y))} classes  |  Grid: {grid_size} configs")
    print(f"{'='*92}")

    for budget in BUDGETS:
        o_scores, a_scores = [], []
        o_times,  a_times  = [], []
        pruning_saved, pruning_possible = 0, 0
        failures = 0

        for s in range(N_SEEDS):
            seed = SEED_BASE + s
            try:
                o_s, o_t = run_optuna(model, grid, X, y, budget, seed)
                a_s, a_t, a_saved, a_poss, _ = run_ags(model, grid, X, y, budget, seed)
                o_scores.append(o_s); o_times.append(o_t)
                a_scores.append(a_s); a_times.append(a_t)
                pruning_saved    += a_saved
                pruning_possible += a_poss
            except Exception as e:
                failures += 1
                print(f"    seed {seed} FAILED: {type(e).__name__}: {e}")

        if not o_scores or not a_scores:
            print(f"\n  BUDGET {budget}: all seeds failed — skipping")
            continue

        o_mean = mean(o_scores)
        a_mean = mean(a_scores)
        delta  = a_mean - o_mean
        a_wins = sum(1 for a, o in zip(a_scores, o_scores) if a > o + 1e-9)
        o_wins = sum(1 for a, o in zip(a_scores, o_scores) if o > a + 1e-9)
        prune_pct = (100 * pruning_saved / pruning_possible
                     if pruning_possible else 0.0)

        print(f"\n  BUDGET {budget}  ({len(o_scores)} seeds, {failures} failed):")
        print(f"    {'Method':<10}{'Mean':>10}{'Std':>10}{'Time(s)':>10}{'Folds':>10}")
        print(f"    {'-'*50}")
        print(f"    {'Optuna':<10}{o_mean:>10.4f}"
              f"{np.std(o_scores, ddof=1):>10.4f}"
              f"{mean(o_times):>10.1f}{budget*CV:>10}")
        a_folds_run = budget*CV - pruning_saved
        print(f"    {'AGS':<10}{a_mean:>10.4f}"
              f"{np.std(a_scores, ddof=1):>10.4f}"
              f"{mean(a_times):>10.1f}{a_folds_run:>10}")
        print(f"    Δ (AGS−Opt) : {delta:+.4f}   "
              f"AGS {a_wins}/{len(o_scores)}  Optuna {o_wins}/{len(o_scores)}  "
              f"pruning {prune_pct:.1f}%")

        all_results.append({
            "dataset": ds_name, "estimator": est_name,
            "space_type": space_type, "budget": budget,
            "o_mean": o_mean, "a_mean": a_mean,
            "o_time": mean(o_times), "a_time": mean(a_times),
            "a_wins": a_wins, "o_wins": o_wins,
            "pruning_pct": prune_pct,
        })


# ============================================================
# PER-ESTIMATOR SUMMARY
# ============================================================
print("\n" + "=" * 92)
print("PER-ESTIMATOR SUMMARY")
print("=" * 92)
print(f"{'Estimator':<16}{'Space type':<26}{'Problems':>9}{'Mean Δ':>10}"
      f"{'AGS wins':>10}{'Opt wins':>10}{'Pruning%':>10}")
print("-" * 92)

for est in sorted(set(r["estimator"] for r in all_results)):
    rows  = [r for r in all_results if r["estimator"] == est]
    space = rows[0]["space_type"]
    deltas = [r["a_mean"] - r["o_mean"] for r in rows]
    a_w = sum(r["a_wins"] for r in rows)
    o_w = sum(r["o_wins"] for r in rows)
    prune = mean(r["pruning_pct"] for r in rows)
    print(f"{est:<16}{space:<26}{len(rows):>9}{mean(deltas):>+10.4f}"
          f"{a_w:>10}{o_w:>10}{prune:>9.1f}%")


# ============================================================
# PER-BUDGET SUMMARY
# ============================================================
print("\n" + "=" * 92)
print("PER-BUDGET SUMMARY")
print("=" * 92)
for b in BUDGETS:
    rows = [r for r in all_results if r["budget"] == b]
    if not rows:
        continue
    deltas = [r["a_mean"] - r["o_mean"] for r in rows]
    a_w = sum(r["a_wins"] for r in rows)
    o_w = sum(r["o_wins"] for r in rows)
    a_fast = sum(1 for r in rows if r["a_time"] < r["o_time"])
    print(f"  Budget {b:>3}: {len(rows)} problems  |  "
          f"mean Δ = {mean(deltas):+.4f}  |  "
          f"AGS {a_w} vs Optuna {o_w}  |  "
          f"AGS faster {a_fast}/{len(rows)}")


# ============================================================
# FINAL AGGREGATE
# ============================================================
print("\n" + "=" * 92)
print("FINAL AGGREGATE  (all problem × budget pairs)")
print("=" * 92)

deltas = np.array([r["a_mean"] - r["o_mean"] for r in all_results])
n = len(deltas)
a_total = sum(r["a_wins"] for r in all_results)
o_total = sum(r["o_wins"] for r in all_results)
mean_delta = float(deltas.mean()) if n else 0.0
std_delta  = float(deltas.std(ddof=1)) if n > 1 else 0.0

print(f"  Problems            : {n}  (dataset × estimator × budget)")
print(f"  Mean Δ (AGS−Opt)    : {mean_delta:+.4f}")
print(f"  Median Δ            : {float(np.median(deltas)):+.4f}" if n else "  Median Δ            : n/a")
print(f"  Std Δ               : {std_delta:.4f}")
print(f"  AGS seed-wins       : {a_total}")
print(f"  Optuna seed-wins    : {o_total}")
print(f"  Problems AGS better : {sum(1 for d in deltas if d >  0.001)}/{n}")
print(f"  Problems Opt better : {sum(1 for d in deltas if d < -0.001)}/{n}")
print(f"  Statistical ties    : {sum(1 for d in deltas if abs(d) <= 0.001)}/{n}")

if HAS_SCIPY and n >= 5:
    stat, p = wilcoxon(deltas)
    print(f"\n  Wilcoxon signed-rank (Δ scores across problems):")
    print(f"    statistic = {stat:.2f}   p = {p:.4f}   "
          f"→ {'SIGNIFICANT' if p < 0.05 else 'NOT significant'} at α=0.05")

o_time_mean = mean(r["o_time"] for r in all_results)
a_time_mean = mean(r["a_time"] for r in all_results)
ratio = o_time_mean / a_time_mean if a_time_mean else float("inf")
print(f"\n  Mean time per problem-budget:")
print(f"    Optuna : {o_time_mean:.1f}s")
print(f"    AGS    : {a_time_mean:.1f}s  ({ratio:.2f}× vs Optuna)")


# ============================================================
# NICHE ANALYSIS
# ============================================================
print("\n" + "=" * 92)
print("NICHE ANALYSIS  (AGS performance by search-space type)")
print("=" * 92)
print(f"{'Space type':<28}{'Problems':>10}{'Mean Δ':>10}{'AGS wins':>10}{'Verdict':>18}")
print("-" * 92)

for space in sorted(set(r["space_type"] for r in all_results)):
    rows = [r for r in all_results if r["space_type"] == space]
    d  = mean(r["a_mean"] - r["o_mean"] for r in rows)
    aw = sum(r["a_wins"] for r in rows)
    if   d >  0.002: verdict = "AGS favours"
    elif d < -0.002: verdict = "Optuna favours"
    else:            verdict = "tie"
    print(f"{space:<28}{len(rows):>10}{d:>+10.4f}{aw:>10}{verdict:>18}")

print("\n" + "=" * 92)
print("Done.")
print("=" * 92)

COMPREHENSIVE MULTI-PROBLEM BENCHMARK — AGS v3.7 vs Optuna TPE
Metric: f1_macro  |  CV: 5  |  Seeds: 5  |  Budgets: [30, 60]
n_jobs: -1 (both)  |  XGBoost available: True
AGS public attributes (probe): ['acquisition', 'acquisition_scores', 'best_params', 'best_score', 'best_state', 'calculate_f', 'candidate_pool_size', 'candidate_pool_size_used', 'climber_patience', 'climbers', 'climbers_merged', 'climbers_spawned', 'completed_evaluations', 'cv', 'distance', 'diversity_resolutions', 'diversity_weight', 'early_stopping_patience', 'enable_pruning', 'estimator', 'evaluate_batch', 'evaluated', 'exploration_weight', 'fit', 'fit_surrogate', 'folds_saved', 'get_neighbors', 'has_conditional_parameters', 'history', 'initial_points', 'initial_trust_radius', 'lazy_search', 'max_evaluations', 'max_neighbor_radius', 'mean_fidelity', 'min_folds_before_pruning', 'min_history_for_percentile', 'min_observations', 'min_trust_radius', 'n_climbers', 'n_completed_evaluations', 'n_evaluations', 'n_jobs', 'n